JOIN CSV

In [1]:
import pandas as pd

fcsv1 = "/mnt/extended-home/dzakaaufa/dataset/caption/data_B_zero_inject.csv"
fcsv2 = "/mnt/extended-home/dzakaaufa/dataset/caption/data_A_inject.csv"

# Load kedua CSV
csv1 = pd.read_csv(fcsv1)
csv2 = pd.read_csv(fcsv2)

# --- NORMALISASI NAMA KOLOM ---
csv1 = csv1.rename(columns={
    "Nama": "name",
    "CLASS": "class",
    "Image Path": "image_path",
    "CAPTION_EN": "caption_en"
})

csv2 = csv2.rename(columns={
    "NAME": "name",
    "CLASS": "class",
    "Image Path": "image_path",
    "CAPTION_EN": "caption_en"
})

# --- PILIH KOLOM YANG DIPERLUKAN ---
csv1_selected = csv1[["name", "class", "image_path", "caption_en"]]
csv2_selected = csv2[["name", "class", "image_path", "caption_en"]]

# --- GABUNGKAN ---
final_df = pd.concat(
    [csv1_selected, csv2_selected],
    ignore_index=True
)

# --- HAPUS DUPLIKAT (OPSIONAL) ---
final_df = final_df.drop_duplicates()

# --- SIMPAN ---
output_path = "/mnt/extended-home/dzakaaufa/dataset/caption/data_mix_noclass1.csv"

final_df.to_csv(output_path, index=False)

# --- INFO DATASET ---
print("=" * 50)
print("CSV berhasil dibuat")
print(f"Lokasi file : {output_path}")
print(f"Total data  : {len(final_df)}")
print("=" * 50)

# --- COUNT PER KELAS ---
print("\nJumlah data per kelas:\n")
print(final_df["class"].value_counts())

CSV berhasil dibuat
Lokasi file : /mnt/extended-home/dzakaaufa/dataset/caption/data_mix_noclass1.csv
Total data  : 3327

Jumlah data per kelas:

class
Malang           162
tribusono        150
buketan          150
sekarjagad       150
wahyu_tumurun    150
jlamprang        150
sidomukti        150
kawung           150
wirasat          150
mega_mendung     150
tuntrum          150
srikaton         150
dayak            150
sidoluhur        150
parang           150
bokor_kencono    150
tujuh_rupa       150
singa_barong     150
liong            150
betawi           150
sidomulyo        150
Lamongan          94
Trenggalek        60
Tulungagung       11
Name: count, dtype: int64


FIX ERROR

In [1]:
import pandas as pd
import os

# ==========================================================
# PATH
# ==========================================================

GT_CSV = "/mnt/extended-home/dzakaaufa/dataset/caption/train_data_caption_noclass.csv"

PRED_CSV = "/mnt/extended-home/dzakaaufa/evaluation_results/batch/Qwen2.5-VL_7B_Batik/noinject/hard1.csv"

OUTPUT_CSV = "/mnt/extended-home/dzakaaufa/evaluation_results/batch/Qwen2.5-VL_7B_Batik/noinject/hard1_fixed.csv"

# ==========================================================
# LOAD
# ==========================================================

gt_df = pd.read_csv(GT_CSV)
pred_df = pd.read_csv(PRED_CSV)

# ==========================================================
# NORMALIZE IMAGE PATH
# ==========================================================

def normalize_filename(path):
    return (
        os.path.basename(str(path))
        .strip()
        .lower()
    )

gt_df["filename"] = gt_df["image_path"].apply(normalize_filename)
pred_df["filename"] = pred_df["image_path"].apply(normalize_filename)

# ==========================================================
# CREATE MAPPING
# ==========================================================

gt_mapping = dict(
    zip(gt_df["filename"], gt_df["caption_en"])
)

# ==========================================================
# REPLACE REFERENCE
# ==========================================================

pred_df["reference"] = pred_df["filename"].map(gt_mapping)

# ==========================================================
# CHECK MISSING
# ==========================================================

missing = pred_df["reference"].isna().sum()

print("=" * 50)
print(f"Total data          : {len(pred_df)}")
print(f"Missing reference   : {missing}")
print("=" * 50)

if missing > 0:
    print("\nContoh data missing:\n")
    print(
        pred_df[pred_df["reference"].isna()][
            ["image_path", "filename"]
        ].head(20)
    )

# ==========================================================
# DROP HELPER COLUMN
# ==========================================================

pred_df = pred_df.drop(columns=["filename"])

# ==========================================================
# SAVE
# ==========================================================

pred_df.to_csv(OUTPUT_CSV, index=False)

print(f"\n[SUKSES] File repaired disimpan ke:")
print(OUTPUT_CSV)

Total data          : 3346
Missing reference   : 19

Contoh data missing:

                                             image_path  \
124   /mnt/extended-home/dzakaaufa/dataset/all_image...   
191   /mnt/extended-home/dzakaaufa/dataset/all_image...   
285   /mnt/extended-home/dzakaaufa/dataset/all_image...   
572   /mnt/extended-home/dzakaaufa/dataset/all_image...   
641   /mnt/extended-home/dzakaaufa/dataset/all_image...   
795   /mnt/extended-home/dzakaaufa/dataset/all_image...   
1179  /mnt/extended-home/dzakaaufa/dataset/all_image...   
1208  /mnt/extended-home/dzakaaufa/dataset/all_image...   
1250  /mnt/extended-home/dzakaaufa/dataset/all_image...   
2256  /mnt/extended-home/dzakaaufa/dataset/all_image...   
2375  /mnt/extended-home/dzakaaufa/dataset/all_image...   
2552  /mnt/extended-home/dzakaaufa/dataset/all_image...   
2558  /mnt/extended-home/dzakaaufa/dataset/all_image...   
2622  /mnt/extended-home/dzakaaufa/dataset/all_image...   
3094  /mnt/extended-home/dzakaaufa/datas

JOIN GAMBAR

In [2]:
import os
import shutil

def merge_all_images(source_dir, output_dir):

    # Buat folder output jika belum ada
    os.makedirs(output_dir, exist_ok=True)

    total_images = 0

    # Telusuri semua folder dan subfolder
    for root, dirs, files in os.walk(source_dir):

        for file_name in files:

            # Filter file gambar
            if file_name.lower().endswith(('.jpg', '.jpeg', '.png')):

                src_path = os.path.join(root, file_name)

                # Hindari nama file bentrok
                base_name, ext = os.path.splitext(file_name)
                new_name = file_name

                counter = 1

                while os.path.exists(os.path.join(output_dir, new_name)):
                    new_name = f"{base_name}_{counter}{ext}"
                    counter += 1

                dst_path = os.path.join(output_dir, new_name)

                # Copy gambar
                shutil.copy2(src_path, dst_path)

                total_images += 1

                print(f"Copied: {new_name}")

    print("\n" + "=" * 50)
    print(f"TOTAL GAMBAR BERHASIL DIGABUNG : {total_images}")
    print(f"HASIL DISIMPAN DI            : {output_dir}")
    print("=" * 50)


# Contoh penggunaan
source_dir = "/mnt/extended-home/dzakaaufa/dataset/image_genbatik"
output_dir = "/mnt/extended-home/dzakaaufa/dataset/all_images_captioning"

merge_all_images(source_dir, output_dir)

Copied: Mulifah Data 8 Modang Gendagan (Api x Membara).jpg
Copied: Data 19 Batik Blimbing (Daniswara Nirbaya).jpg
Copied: Data 3 Batik Krajan (Bunga Terompet Garudeya).jpg
Copied: Data 12 Batik Rahayu (Bledak Bouket).jpg
Copied: Data 13 Batik Krajan (Garudeya Daun).jpg
Copied: Data 9 Batik Wagastu (Bon Kelengkeng).jpg
Copied: Data 14 Batik Krajan (Parang Kopi).jpg
Copied: Tutut Data 7 Bandeng Lele Wijaya Kusuma.jpg
Copied: Mutiara S Data 2 Mawar Bersulur.jpg
Copied: Data 2 Batik Baronggung (Jahewono).jpg
Copied: Data 31 Batik Kantil (Mawar).jpg
Copied: Data 7 Batik Bengkel (Mayuragari).jpg
Copied: Istiqomah Data 4 Pisang Latohan.jpg
Copied: Jayyida Data 14 Singa Duduk (Mitos).jpg
Copied: Afiq Jaya_35241420061002_Gapuro Teratai_2025_FLORA_FAUNA.jpg
Copied: Mutiara S Data 1 Bandeng Lele dalam Kolam.jpg
Copied: Data 5 Batik Tiepuk (Kipas Cengkeh).jpg
Copied: Data 5 Batik Krajan (Bunga Sepatu).jpg
Copied: Data 14 Batik Soendari (Bapang Bonpring).jpg
Copied: Data 11 Batik Baronggung (Rumput

JOIN TRAIN TEST VAL

In [2]:
import os
import shutil

def merge_dataset(source_dir, output_dir):

    splits = ['train', 'val', 'test']

    total_all_images = 0

    for split in splits:

        split_path = os.path.join(source_dir, split)

        split_count = 0

        # Loop semua kelas
        for class_name in os.listdir(split_path):

            class_path = os.path.join(split_path, class_name)

            if os.path.isdir(class_path):

                # Folder tujuan
                target_class_dir = os.path.join(output_dir, class_name)
                os.makedirs(target_class_dir, exist_ok=True)

                # Ambil file gambar
                image_files = [
                    f for f in os.listdir(class_path)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
                ]

                class_count = 0

                for image_name in image_files:

                    src_image = os.path.join(class_path, image_name)

                    # Hindari overwrite
                    new_name = f"{split}_{image_name}"

                    dst_image = os.path.join(target_class_dir, new_name)

                    shutil.copy2(src_image, dst_image)

                    class_count += 1
                    split_count += 1
                    total_all_images += 1

                print(f"[{split}] {class_name} : {class_count} gambar")

        print(f"\nTOTAL {split.upper()} : {split_count} gambar\n")

    print("=" * 50)
    print(f"TOTAL SELURUH DATASET : {total_all_images} gambar")
    print("=" * 50)
    print("SEMUA DATASET BERHASIL DIGABUNG")


# Contoh penggunaan
source_dir = "/mnt/extended-home/dzakaaufa/dataset/baru_captioning"
output_dir = "/mnt/extended-home/dzakaaufa/dataset/path"

merge_dataset(source_dir, output_dir)

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/extended-home/dzakaaufa/dataset/baru_captioning/train'

split train test val

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

def simple_stratified_split(csv_path, output_path):
    df = pd.read_csv(csv_path)

    # Split pertama: 80% train, 20% sisa (val + test)
    # 20% sisa tersebut kemudian dibagi dua menjadi 10% val dan 10% test
    train_df, temp_df = train_test_split(
        df, test_size=0.2, stratify=df['class'], random_state=42
    )
    
    val_df, test_df = train_test_split(
        temp_df, test_size=0.5, stratify=temp_df['class'], random_state=42
    )

    # Tandai split
    train_df = train_df.assign(split='train')
    val_df = val_df.assign(split='val')
    test_df = test_df.assign(split='test')

    # Gabung dan simpan
    master_df = pd.concat([train_df, val_df, test_df])
    master_df.to_csv(output_path, index=False)

    print(f"Split selesai: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

# Jalankan fungsi
simple_stratified_split(
    "/mnt/extended-home/dzakaaufa/dataset/caption/data_train_dino.csv", 
    "/mnt/extended-home/dzakaaufa/dataset/caption/data_train_dino_split.csv"
)

Split selesai: Train=2661, Val=333, Test=333


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split

def create_advanced_lecturer_split(csv_path, output_path):
    df = pd.read_csv(csv_path)
    
    # Normalisasi nama kolom agar seragam (Antisipasi kapitalisasi)
    df = df.rename(columns={
        "Nama": "name", "NAME": "name",
        "CLASS": "class", "Class": "class", "CLASS ": "class",
        "Image Path": "image_path", "IMAGE_PATH": "image_path",
        "CAPTION_EN": "caption_en", "Caption_En": "caption_en"
    })
    
    # Pisahkan Dataset A (4 kelas asli) dan Dataset B (20 kelas otomatis)
    # Deteksi berdasarkan nama kelas unik di Dataset A
    kelas_a = ['tulungagung', 'trenggalek', 'malang', 'lamongan']
    
    df_a = df[df['class'].str.lower().isin(kelas_a)].copy()
    df_b = df[~df['class'].str.lower().isin(kelas_a)].copy()
    
    print(f"Deteksi Awal: Dataset A = {len(df_a)} data, Dataset B = {len(df_b)} data")
    
    # ==========================================
    # PROSES SPLIT DATASET A (Total 346)
    # ==========================================
    # 1. Ambil 150 data untuk TEST secara stratified berdasarkan kelas
    # (150 / 346 = ~0.433)
    test_size_a = 150 / len(df_a)
    temp_df_a, test_df_a = train_test_split(
        df_a, test_size=test_size_a, stratify=df_a['class'], random_state=42
    )
    
    # 2. Sisa Data A (196 data) di-split untuk Train dan Val (misal 33 data untuk Val, sisanya Train)
    # 33 / 196 = ~0.168
    val_size_a = 33 / len(temp_df_a)
    train_df_a, val_df_a = train_test_split(
        temp_df_a, test_size=val_size_a, stratify=temp_df_a['class'], random_state=42
    )
    
    # ==========================================
    # PROSES SPLIT DATASET B (Total 3.000)
    # ==========================================
    # Ditumpuk ke Train (2700) dan Val (300) -> Rasio 10% untuk Val
    train_df_b, val_df_b = train_test_split(
        df_b, test_size=0.1, stratify=df_b['class'], random_state=42
    )
    
    # ==========================================
    # PEMBERIAN LABEL DAN PENGGABUNGAN
    # ==========================================
    train_df_a['split'] = 'train'; train_df_b['split'] = 'train'
    val_df_a['split'] = 'val'; val_df_b['split'] = 'val'
    test_df_a['split'] = 'test' # Dataset B tidak menyumbang ke Test set
    
    final_train = pd.concat([train_df_a, train_df_b])
    final_val = pd.concat([val_df_a, val_df_b])
    final_test = test_df_a
    
    # Gabungkan semua menjadi satu berkas master
    master_df = pd.concat([final_train, final_val, final_test], ignore_index=True)
    master_df.to_csv(output_path, index=False)
    
    print("\n" + "="*50)
    print(" PIPELINE SPLIT DATASET BERHASIL DI-GENERATE")
    print("="*50)
    print(f"Total DATA TRAIN (A+B) : {len(final_train)}")
    print(f"Total DATA VAL   (A+B) : {len(final_val)}")
    print(f"Total DATA TEST  (A)   : {len(final_test)}")
    print(f"Grand Total Dataset    : {len(master_df)}")
    print("="*50)

# Jalankan fungsi
create_advanced_lecturer_split(
    "/mnt/extended-home/dzakaaufa/dataset/caption/data_mix_inject1.csv", 
    "/mnt/extended-home/dzakaaufa/dataset/caption/data_mix_inject_split1.csv"
)

Deteksi Awal: Dataset A = 327 data, Dataset B = 3000 data

 PIPELINE SPLIT DATASET BERHASIL DI-GENERATE
Total DATA TRAIN (A+B) : 2844
Total DATA VAL   (A+B) : 333
Total DATA TEST  (A)   : 150
Grand Total Dataset    : 3327
